# Perceived employability of senior executives

Individual Machine Learning course project by Sudhanshu Dubey.

This public copy keeps the original analysis code and Markdown. Saved cell outputs and execution metadata have been cleared. The restricted course dataset is not included. The interactive dashboard at the repository root presents selected aggregate results.

To rerun the analysis, you must already have authorised access to the original `employability.csv`. Place it in `data/`, keep the notebook working directory as `notebook/`, install `requirements.txt`, and run the cells in order. See the README and DATA_NOTICE.md.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

DATA_DIR = Path('../data')
df = pd.read_csv(DATA_DIR / 'employability.csv')

In [ ]:
df.info()

In [ ]:
%pip install pandas numpy matplotlib seaborn scipy pingouin

In [ ]:
import seaborn as sns
import pingouin as pg

from scipy.stats import chi2_contingency

In [ ]:
def UniCat(df, col):
    counts = df[col].value_counts(dropna=False)
    proportions = df[col].value_counts(normalize=True, dropna=False)

    summary = pd.DataFrame({
        'Frequency': counts,
        'Proportion': proportions
    })

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    counts.plot(kind='bar', ax=axes[0])
    axes[0].set_title('Bar Chart')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Frequency')

    counts.plot(kind='pie', autopct='%1.1f%%', ax=axes[1])
    axes[1].set_title('Pie Chart')
    axes[1].set_ylabel('')

    plt.tight_layout()
    plt.show()

    return summary

In [ ]:
job_status_summary = UniCat(df, 'job_status')
job_status_summary

In [ ]:
sex_summary = UniCat(df, 'sex')
sex_summary

In [ ]:
education_summary = UniCat(df, 'education_level')
education_summary

In [ ]:
degree_type_summary = UniCat(df, 'degree_type')
degree_type_summary

In [ ]:
education_place_summary = UniCat(df, 'education_place')
education_place_summary

In [ ]:
sector_summary = UniCat(df, 'sector')
sector_summary

In [ ]:
family_status_summary = UniCat(df, 'family_status')
family_status_summary

In [ ]:
salary_summary = UniCat(df, 'salary_band')
salary_summary

In [ ]:
def UniQuant(df, col, bins=10):
    values = df[col].dropna()

    summary = values.describe().loc[['min', '25%', '50%', '75%', 'max']]

    Q1 = summary['25%']
    Q3 = summary['75%']
    IQR = Q3 - Q1

    summary.loc['Lower Fence'] = Q1 - 1.5 * IQR
    summary.loc['Upper Fence'] = Q3 + 1.5 * IQR

    intervals = pd.cut(values, bins=bins, right=False)
    frequencies = intervals.value_counts().sort_index()

    freq_table = pd.DataFrame({
        'Frequency': frequencies,
        'Proportion': frequencies / frequencies.sum()
    })

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].hist(values, bins=bins, edgecolor='black')
    axes[0].set_title('Histogram')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Frequency')

    sns.boxplot(x=values, ax=axes[1])
    axes[1].set_title('Boxplot')

    plt.tight_layout()
    plt.show()

    return summary, freq_table

In [ ]:
age_summary, age_frequency = UniQuant(df, 'age', bins=5)

display(age_summary)
display(age_frequency)

In [ ]:
children_summary, children_frequency = UniQuant(df, 'children', bins=5)

display(children_summary)
display(children_frequency)

In [ ]:
self_efficacy_summary, self_efficacy_frequency = UniQuant(df, 'self_efficacy', bins=6)

display(self_efficacy_summary)
display(self_efficacy_frequency)

In [ ]:
competence_summary, competence_frequency = UniQuant(df, 'competence', bins=6)

display(competence_summary)
display(competence_frequency)

In [ ]:
anticipation_summary, anticipation_frequency = UniQuant(df, 'anticipation', bins=6)

display(anticipation_summary)
display(anticipation_frequency)

In [ ]:
flexibility_summary, flexibility_frequency = UniQuant(df, 'flexibility', bins=6)

display(flexibility_summary)
display(flexibility_frequency)

In [ ]:
job_search_summary, job_search_frequency = UniQuant(df, 'job_search_techniques', bins=6)

display(job_search_summary)
display(job_search_frequency)

In [ ]:
training_summary, training_frequency = UniQuant(df, 'training', bins=6)

display(training_summary)
display(training_frequency)


In [ ]:
alternative_summary, alternative_frequency = UniQuant(df, 'alternative_employment', bins=6)

display(alternative_summary)
display(alternative_frequency)

In [ ]:
remaining_scores = [
    'value_awareness',
    'market_knowledge',
    'versatility',
    'know_how_visibility',
    'networks'
]

for score in remaining_scores:
    print(score)

    summary, frequency = UniQuant(df, score, bins=6)

    display(summary)
    display(frequency)

In [ ]:
score_cols = [
    'competence',
    'anticipation',
    'flexibility',
    'job_search_techniques',
    'self_efficacy',
    'training',
    'alternative_employment',
    'value_awareness',
    'market_knowledge',
    'versatility',
    'know_how_visibility',
    'networks'
]

scores = df[score_cols]
score_corr = scores.corr(method='pearson')

fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(
    score_corr, annot=True, fmt='.2f',
    cmap='coolwarm', vmin=-1, vmax=1, ax=ax
)

ax.set_title('Correlations between employability scores')
plt.tight_layout()
plt.show()

In [ ]:
anova_status = pg.anova(
    data=df,
    dv='self_efficacy',
    between='job_status',
    detailed=True
)

ss_between = anova_status.loc[0, 'SS']
ss_within = anova_status.loc[1, 'SS']

anova_status.loc[0, 'eta2'] = ss_between / (ss_between + ss_within)

display(anova_status)

sns.boxplot(data=df, x='job_status', y='self_efficacy')
plt.title('Self-efficacy by job status')
plt.show()

In [ ]:
for col in ['age', 'children']:
    print(col)

    correlation = pg.pairwise_corr(
        data=df[['self_efficacy', col]],
        method='pearson',
        padjust=None
    )

    display(correlation)

In [ ]:
observed = pd.crosstab(df['sex'], df['job_status'])

chi2, p_value, dof, expected = chi2_contingency(
    observed, correction=False
)

n = observed.values.sum()
rows, cols = observed.shape
cramers_v = np.sqrt(chi2 / (n * min(rows - 1, cols - 1)))

display(observed)

print('Chi-square:', chi2)
print('Degrees of freedom:', dof)
print('p-value:', p_value)
print("Cramer's V:", cramers_v)
print('Minimum expected count:', expected.min())

observed.plot(kind='bar')
plt.title('Job status by sex')
plt.ylabel('Number of respondents')
plt.tight_layout()
plt.show()

In [ ]:
for score in ['training', 'networks']:
    print(score)

    anova_result = pg.anova(
        data=df,
        dv=score,
        between='job_status',
        detailed=True
    )

    ss_between = anova_result.loc[0, 'SS']
    ss_within = anova_result.loc[1, 'SS']
    anova_result.loc[0, 'eta2'] = ss_between / (ss_between + ss_within)

    display(anova_result)

    print('Employed mean:',
          df.loc[df['job_status'] == 'Employed', score].mean())
    print('Job seeker mean:',
          df.loc[df['job_status'] == 'Job seeker', score].mean())

    sns.boxplot(data=df, x='job_status', y=score)
    plt.title(score + ' by job status')
    plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(df['age'], df['self_efficacy'])
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Self-efficacy')

axes[1].scatter(df['children'], df['self_efficacy'])
axes[1].set_xlabel('Dependent children (4 = 4 or more)')
axes[1].set_ylabel('Self-efficacy')

plt.tight_layout()
plt.show()

**Part 1 — Exploratory analysis**

Part 1 explored the 460 respondents and 12 employability scores using descriptive tables and plots. No missing values were detected, and rare categories were identified. Relationships were examined using Pearson correlations, ANOVA with η², and chi-square tests with Cramér’s V. Competence had the highest median and alternative employment the lowest. Self-efficacy was most closely correlated with value awareness and flexibility, and weakly negatively correlated with age. Job seekers reported higher training and network scores. These findings describe associations and do not establish causation.

# Part 2 — Preparing the data

In [ ]:
df_prepared = df.copy()

df_prepared = df_prepared.drop(columns=['respondent'])

# Combine the two smallest sectors with Other
df_prepared['sector'] = df_prepared['sector'].replace({
    'Real estate': 'Other',
    'Construction': 'Other'
})

# Group the salary bands
df_prepared['salary_band'] = df_prepared['salary_band'].replace({
    '< 20k': '< 60k',
    '20-29k': '< 60k',
    '30-39k': '< 60k',
    '40-49k': '< 60k',
    '50-59k': '< 60k',
    '60-69k': '60-99k',
    '70-79k': '60-99k',
    '80-89k': '60-99k',
    '90-99k': '60-99k'
})

In [ ]:
print(df_prepared.shape)

display(df_prepared['sector'].value_counts())
display(df_prepared['salary_band'].value_counts())

In [ ]:
df_encoded = pd.get_dummies(
    df_prepared,
    drop_first=True,
    dtype=float
)

df_encoded.info()

*Part 2 — Preparation Choices*

The `respondent` column was excluded because it is only an identifier. In the `sector` variable, Real estate and Construction were combined with Other because they had only 9 and 7 respondents, respectively. This reduces some detail but avoids working with very small groups. Salary bands were grouped into three categories: `< 60k`, `60–99k`, and `> 99k` to simplify the analysis while keeping the main salary differences. Education levels were kept separate, although the small Bachelor group and the limited variation in `education_place` were noted for later predictor selection. Categorical variables were converted into dummy variables, with one reference category for each, while numerical variables were left unchanged. Since there were no missing values, no imputation was needed, and all 460 respondents were retained.

# Part 3 – What drives professional self-efficacy?

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
y = df_encoded['self_efficacy']
X = df_encoded.drop(columns=['self_efficacy'])

model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)

print('X:', X.shape)
print('y:', y.shape)

In [ ]:
n = len(y)
p = X.shape[1]

SSE = np.sum((y - y_pred)**2)
SST = np.sum((y - y.mean())**2)

rmse = np.sqrt(SSE / n)
r2 = 1 - SSE / SST
r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f'RMSE: {rmse:.4f}')
print(f'R²: {r2:.4f}')
print(f'Adjusted R²: {r2_adj:.4f}')

In [ ]:
coefficients = model.coef_
std_coef = coefficients * X.std().values / y.std()

coef_table = pd.DataFrame({
    'Coefficient': coefficients,
    'Standardized Coefficient': std_coef
}, index=X.columns)

print(f'Intercept: {model.intercept_:.4f}')
display(coef_table)

In [ ]:
score_predictors = [
    'competence', 'anticipation', 'flexibility',
    'job_search_techniques', 'training', 'alternative_employment',
    'value_awareness', 'market_knowledge', 'versatility',
    'know_how_visibility', 'networks'
]

X_scores = df_encoded[score_predictors]

model_scores = LinearRegression()
model_scores.fit(X_scores, y)
y_pred_scores = model_scores.predict(X_scores)

n = len(y)
p_scores = X_scores.shape[1]

SSE_scores = np.sum((y - y_pred_scores)**2)
SST_scores = np.sum((y - y.mean())**2)

rmse_scores = np.sqrt(SSE_scores / n)
r2_scores = 1 - SSE_scores / SST_scores
r2_adj_scores = 1 - (1 - r2_scores) * (n - 1) / (n - p_scores - 1)

comparison = pd.DataFrame({
    'Predictors': [p_scores, X.shape[1]],
    'RMSE': [rmse_scores, rmse],
    'R²': [r2_scores, r2],
    'Adjusted R²': [r2_adj_scores, r2_adj]
}, index=['Scores only', 'Scores + personal variables'])

display(comparison.round(4))

We fitted a multiple linear regression explaining self-efficacy using the other 11 employability scores and the prepared personal variables. The respondent identifier was excluded because it does not describe a respondent’s characteristics. The full model achieved R² = 0.4620, adjusted R² = 0.4230 and RMSE = 1.0785. Compared with the scores-only model, adding personal variables increased the variation accounted for by approximately 6 percentage points.

Holding the other predictors constant, a one-point increase in flexibility was associated with 0.551 additional self-efficacy points, while a one-point increase in value awareness was associated with 0.434 points. These variables also had the largest absolute standardised coefficients, at 0.373 and 0.322. Their ranking depends on the other included predictors because several scores are correlated. These results describe associations and fit to the observed sample; they do not establish causation or performance on new respondents.

# Part 4 – Who are the job seekers?

In [ ]:
# 1 = Job seeker, 0 = Employed
y_job = df_encoded['job_status_Job seeker'].astype(int)

display(y_job.value_counts().sort_index())

baseline = y_job.value_counts(normalize=True).max()
print(f'Majority-class baseline accuracy: {baseline:.2%}')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
X_job = df_encoded.drop(columns=['job_status_Job seeker'])

X_train, X_valid, y_train, y_valid = train_test_split(
    X_job, y_job,
    test_size=0.20,
    random_state=42,
    stratify=y_job
)

logreg = LogisticRegression(C=np.inf, solver='newton-cholesky')
logreg.fit(X_train, y_train)

print('Training:', X_train.shape)
print('Validation:', X_valid.shape)

In [ ]:
y_job_pred = logreg.predict(X_valid)

accuracy_valid = logreg.score(X_valid, y_valid)
baseline_valid = (y_valid == 0).mean()

print(f'Validation accuracy: {accuracy_valid:.2%}')
print(f'Baseline accuracy: {baseline_valid:.2%}')

cm = confusion_matrix(y_valid, y_job_pred, labels=[0, 1])
ConfusionMatrixDisplay(
    cm, display_labels=['Employed', 'Job seeker']
).plot()
plt.show()

print(classification_report(
    y_valid, y_job_pred,
    labels=[0, 1],
    target_names=['Employed', 'Job seeker'],
    digits=3
))

In [ ]:
job_coef = pd.Series(logreg.coef_[0], index=X_train.columns)
job_odds_ratios = np.exp(job_coef)

odds_table = pd.DataFrame({
    'Coefficient': job_coef,
    'Odds ratio': job_odds_ratios
})

display(odds_table.round(4))